# Lab 31 · Publish Jev decision models for typed answers

Publish TypeSafe's Jev decision model through the gateway with a Custom provider, triage support tickets with typed
answers and confidences over REST and with the TypeSafe Python SDK, check how the alias rejects the wrong calls, then
watch a request rate limit stop a burst of decisions. This notebook is the Python track of the
[lab guide](README.md): it deploys the same [infra/jev-decisions.bicep](infra/jev-decisions.bicep) as the
Bicep + azd track, sends the same [infra/ticket.json](infra/ticket.json), and runs the same code as
[infra/triage.py](infra/triage.py).

**Publish Jev → Send a decision request → Triage with the SDK → Wrong calls → Burst → Inspect 429 → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Model provider `typesafe` | A Custom provider for `https://api.typesafe.ai`. Its one authentication header, `Authorization: Bearer <TypeSafe key>`, carries your TypeSafe key on every call to TypeSafe. The gateway never returns the key. |
| Backend model `typesafe/jev-latest` | The Jev model, with one supported endpoint, `typesafe/v1/systemone`, the System One decisions API. |
| Alias `ticket-triage` | The name that clients send in the request `model` field, with a `requestRateLimit` policy: 5 decision requests per minute for each caller identity. |
| API key `app-ticket-triage` | The triage app's runtime key, so that its requests have their own rate-limit counter. |
| [infra/jev-decisions.bicep](infra/jev-decisions.bicep) | Declares the four resources above on the gateway from notebook 01. |
| [infra/triage.py](infra/triage.py) | The sample tickets, the three typed questions, and the TypeSafe SDK client that points at the gateway. |
| [../shared/utils.py](../shared/utils.py) | Signs in with your `az login`, finds the gateway, deploys Bicep, and cleans up. |

## Prerequisites

- The gateway from [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb). This notebook
  doesn't need a Foundry account.
- A TypeSafe API key from the [TypeSafe console](https://console.typesafe.ai/keys). Set it in the `JEV_API_KEY`
  environment variable before you start Jupyter, or paste it when step 0 asks for it. TypeSafe bills the decisions
  that this notebook sends to your TypeSafe account.
- `az login`, Python 3.10 or later, and permission to change the gateway, such as **Contributor** on its resource
  group.

Select **Run All**. The notebook takes about five minutes, including two one-minute waits for a fresh rate-limit
window.

In [ ]:
%pip install -q --upgrade azure-identity requests -r infra/requirements.txt

### 0. Initialize the lab

The defaults match notebook 01, so `utils.Lab` finds the gateway without edits. The cell reads your TypeSafe key from
`JEV_API_KEY`, or asks for it without showing what you type. It never prints the key.

In [ ]:
import os
import sys
from getpass import getpass
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("31-jev-decisions")
alias      = "ticket-triage"       # the alias that the triage app sends in "model"
key_name   = "app-ticket-triage"   # the triage app's runtime key
per_minute = 5                     # decision requests per minute for each caller identity

jev_api_key = os.environ.get("JEV_API_KEY") or getpass("TypeSafe API key: ")
display(brand.note(f"TypeSafe key: {len(jev_api_key)} characters, not shown", "ok"))

### 1. Publish Jev behind an alias

[infra/jev-decisions.bicep](infra/jev-decisions.bicep) creates the four resources in order:

- The Custom provider `typesafe`, whose base URL is `https://api.typesafe.ai`. Its `authentication.headers` holds
  exactly one header, `Authorization`, with the value `Bearer <TypeSafe key>` from a `@secure()` parameter. A provider
  with a second header fails with `Model-provider authentication requires exactly one API-key header.`
- The backend model `jev-latest`, whose `supportedEndpoints` is `["typesafe/v1/systemone"]`, without a leading slash.
- The alias `ticket-triage`, with a `requestRateLimit` of 5 calls per `minute`, counted per `identity`.
- The runtime key `app-ticket-triage`. The template never outputs its value: the cell reads it with `listSecrets`.

In [ ]:
published = lab.deploy("infra/jev-decisions.bicep", gatewayName=lab.gateway_name, jevApiKey=jev_api_key,
                       aliasName=alias, requestsPerMinute=per_minute, apiKeyName=key_name)
provider_id = f"{lab.workspace_id}/modelProviders/{published['providerName']}"
provider    = lab.arm("GET", provider_id)["properties"]
model       = lab.arm("GET", f"{provider_id}/models/{published['modelName']}")["properties"]
policies    = lab.arm("GET", f"{lab.workspace_id}/aliases/{alias}")["properties"]["policies"]

display(brand.info(f"Model provider {published['providerName']} and backend model {published['modelName']}", [
    ("Provider type",       brand.mono(provider["modelProviderType"])),
    ("Base URL",            brand.mono(provider["custom"]["endpoint"]["url"])),
    ("Auth header",         brand.mono(", ".join(h["name"] for h in provider["authentication"]["headers"])
                                       + " (the value is never returned)")),
    ("Supported endpoints", brand.chips(*(brand.chip(e, "violet", code=True) for e in model["supportedEndpoints"]))),
    ("TypeSafe model",      brand.mono(model["deployment"]["modelName"])),
    ("Decisions URL",       brand.mono(published["decisionsUrl"])),
    ("SDK base URL",        brand.mono(published["sdkBaseUrl"])),
]))
display(brand.policies(alias, policies))

api_key = lab.arm("POST", f"{lab.gateway_id}/apiKeys/{key_name}/listSecrets")["primaryKey"]
display(brand.key(key_name, api_key))

assert provider["modelProviderType"] == "Custom" and len(provider["authentication"]["headers"]) == 1
assert "value" not in provider["authentication"]["headers"][0]
assert model["supportedEndpoints"] == ["typesafe/v1/systemone"]
assert policies[0]["type"] == "requestRateLimit" and policies[0]["count"] == per_minute

### 2. Send a decision request

The gateway serves System One at `<gateway>/default/models/typesafe/v1/systemone`. The request is TypeSafe's own
format, with two differences from a direct call: the URL is the gateway's, and the credential is your gateway key
instead of the TypeSafe key. `model` is the alias. The body comes from [infra/ticket.json](infra/ticket.json): one
ticket as the `state`, and three typed questions:

- `urgent` is a `noul`: the answer is the probability of yes, from 0 to 1.
- `team` is a `choice` between `billing`, `technical`, and `account`: the answer is the option, its `confidence`,
  and the `probabilities` of every option.
- `frustration` is a `score` on a three-level rubric: the answer is a level, which can fall between two levels, with
  its `confidence` and a `legend`.

A new alias reaches the runtime within a minute, so the cell sends the request again every 15 seconds until it
answers `200`.

In [ ]:
import json
import requests

decisions_url = published["decisionsUrl"]
auth   = {"Authorization": f"Bearer {api_key}"}
ticket = json.load(open("infra/ticket.json"))

def decide(body, headers=auth):
    """POST a System One request to the gateway as uncompressed JSON."""
    return requests.post(decisions_url, json=body, headers=headers, timeout=60)

sent = {}
def send_ticket():
    sent["response"] = decide({"model": alias, **ticket})
    return sent["response"].status_code

utils.wait_for_status(send_ticket, 200, wait=15)
result  = sent["response"].json()
answers = result["answers"]
level   = answers["frustration"]["legend"][str(round(answers["frustration"]["score"]))]

display(brand.table("Typed answers", ["Question", "Type", "Answer", "Confidence"], [
    ("urgent",      "noul",   f"{answers['urgent']['noul']:.2f} probability of yes", "-"),
    ("team",        "choice", answers["team"]["choice"],                             answers["team"]["confidence"]),
    ("frustration", "score",  f"{answers['frustration']['score']} · {level}",        answers["frustration"]["confidence"]),
], sub=f"{ticket['state']['ticket']['subject']} · answered by {result['model']} · "
       f"{result['usage']['input_tokens']} input and {result['usage']['output_tokens']} output tokens"))

assert set(answers) == {"urgent", "team", "frustration"} and answers["team"]["choice"] == "billing"

### 3. Triage tickets with the TypeSafe SDK

The TypeSafe Python SDK calls the gateway when you give it the gateway's System One base URL,
`<gateway>/default/models/typesafe` (the SDK adds `/v1/systemone`), your gateway key, and the alias. Turn its request
compression off: the gateway takes only uncompressed JSON and answers a compressed body with `415`.

The cell imports [infra/triage.py](infra/triage.py) and triages its three sample tickets with the same three
questions, typed as `Noul`, `Choice`, and `Score`. The first call to a model can take several seconds, and the next
ones about half a second.

In [ ]:
import time

os.environ.update(AI_GATEWAY_URL=lab.gateway_url, AI_GATEWAY_API_KEY=api_key, AI_GATEWAY_MODEL=alias)
sys.path.insert(0, "infra")
import triage

rows, teams = [], []
with triage.make_client() as client:
    for item in triage.TICKETS:
        start = time.perf_counter()
        triaged = client.system_one(state={"ticket": item}, questions=triage.QUESTIONS)
        ms = (time.perf_counter() - start) * 1000
        team, score = triaged.choices["team"], triaged.scores["frustration"]
        teams.append(team.choice)
        rows.append((item["subject"], round(triaged.nouls["urgent"].noul, 2), team.choice, team.confidence,
                     f"{score.score:.1f} · {score.legend[round(score.score)]}", round(ms),
                     f"{triaged.usage.input_tokens} in, {triaged.usage.output_tokens} out"))

display(brand.table("Ticket triage through the gateway",
                    ["Ticket", "Urgent", "Team", "Confidence", "Frustration", "ms", "Tokens"], rows,
                    sub=f"{alias} · answered by {triaged.model} · TypeSafe SDK"))
assert teams == ["billing", "technical", "account"]

### 4. Call the alias the wrong way

The alias serves only the System One endpoint, and only with a gateway credential. Each wrong call gets a clear
status: a Chat Completions request to the alias gets `404` with `SystemOneRequestNotAdmitted`, the backend model's
name instead of the alias gets `404` with `model_not_found`, a missing key gets `401`, and a key in both the
`api-key` and the `Authorization` headers gets `400`. And the TypeSafe SDK with zstd compression, its default on
Python 3.14 or later for bodies of 1 KB or more, gets `415` with `SystemOneEncodingUnsupported`. Without zstd support,
the cell skips that check.

The earlier steps used most of this minute's decision requests, so the cell first waits a minute for a fresh
rate-limit window.

In [ ]:
from typesafe_sdk import RetryPolicy, TypeSafeAPIError, TypeSafeClient, TypeSafeError

def detail(response):
    body = response.json()
    if "answers" in body:
        return f"{len(body['answers'])} answers from {body['model']}"
    error = body.get("error") or {}
    return f"{error.get('code') or body.get('statusCode')}: {error.get('message') or body.get('message')}"

brand.wait(60, "for a fresh rate-limit window")
small = {"model": alias, "state": ticket["state"], "questions": {"urgent": ticket["questions"]["urgent"]}}
chat  = requests.post(f"{lab.gateway_url}/default/models/openai/v1/chat/completions", headers=auth, timeout=60,
                      json={"model": alias, "messages": [{"role": "user", "content": "Is this ticket urgent?"}]})
backend_name = decide({**small, "model": "jev-latest"})
without_key  = decide(small, headers={})
both_headers = decide(small, headers={**auth, "api-key": api_key})

# The SDK compresses only bodies of 1 KB or more: this triage request adds the customer's earlier messages.
thread = {"ticket": ticket["state"]["ticket"],
          "earlier_messages": [f"Message {n}: {ticket['state']['ticket']['message']}" for n in (1, 2, 3)]}
try:
    compressed = TypeSafeClient(api_key=api_key, base_url=published["sdkBaseUrl"], model=alias,
                                compression="zstd", retry=RetryPolicy(max_retries=0))
except TypeSafeError:   # Python before 3.14 without the typesafe-sdk[zstd] extra can't compress
    compressed = None
if compressed:
    try:
        compressed.system_one(state=thread, questions=triage.QUESTIONS)
        compressed_status, compressed_detail = 200, "accepted"
    except TypeSafeAPIError as error:
        compressed_status, compressed_detail = error.status, json.dumps(error.body)
    finally:
        compressed.close()

display(brand.checks("Runtime checks", [
    (f"Chat Completions → {alias}",            chat.status_code,         404, detail(chat)),
    ("System One → jev-latest",                backend_name.status_code, 404, detail(backend_name)),
    ("System One without a key",               without_key.status_code,  401, detail(without_key)),
    ("System One with api-key and Bearer",     both_headers.status_code, 400, detail(both_headers)),
] + ([("TypeSafe SDK with zstd compression", compressed_status, 415, compressed_detail)] if compressed else [])))
assert (chat.status_code, backend_name.status_code, without_key.status_code, both_headers.status_code) == \
    (404, 404, 401, 400)
assert not compressed or compressed_status == 415

### 5. Send a burst of decisions

Every decision costs money at TypeSafe, and a triage job stuck in a loop can send thousands. The alias's request rate
limit gives each caller identity 5 decision requests per minute. After another minute's wait, the cell sends eight
small decisions in a row with the same key. The gateway answers the requests over the limit itself, with `429` and a
`Retry-After` header, so they never reach TypeSafe. In this lab's runs, a sixth request passed before the first
`429`.

In [ ]:
brand.wait(60, "for a fresh rate-limit window")
board = brand.Board(f"Decision burst against {alias}", ["Request", "Status", "Retry-After", "urgent"],
                    sub=f"{key_name} · {per_minute} requests per minute")
burst = []
for number in range(1, 9):
    response = decide(small)
    urgent = response.json()["answers"]["urgent"]["noul"] if response.ok else "-"
    board.add([number, brand.status(response.status_code), response.headers.get("retry-after", "-"), urgent],
              response.status_code)
    burst.append(response)

throttled = [r for r in burst if r.status_code == 429]
assert burst[0].status_code == 200 and throttled, "No 429: check that the alias lists the requestRateLimit policy."

### 6. Inspect a throttled response

Look at the status, headers, and body of one throttled request. `Retry-After` tells the client how many seconds to
wait. The body has `statusCode` and `message`, not TypeSafe's error format, so check the HTTP status and
`Retry-After`. The TypeSafe SDK raises `TypeSafeAPIError` with `status` 429, and with its default retries it waits
for `Retry-After` and sends the request again.

In [ ]:
last = throttled[-1]
display(brand.http(last.status_code, last.headers, last.text, "A throttled decision request", sub=alias,
                   keys=["retry-after", "content-type"]))
assert last.status_code == 429 and "Rate limit is exceeded" in last.text

### 7. Clean up resources

The cell deletes the alias first, then the backend model that it routes to, then the provider with your TypeSafe key,
and the runtime key. The gateway from notebook 01 stays for the next notebooks. Deleting the provider doesn't revoke
your TypeSafe key: if you created it for this lab, delete it in the [TypeSafe console](https://console.typesafe.ai/keys).

In [ ]:
lab.delete(f"{lab.workspace_id}/aliases/{alias}")
lab.delete(f"{provider_id}/models/{published['modelName']}")
lab.delete(provider_id)
lab.delete(f"{lab.gateway_id}/apiKeys/{key_name}")

## Troubleshooting

- **Step 1 fails with `Model-provider authentication requires exactly one API-key header.`**: the provider has more
  than one authentication header. Keep only `Authorization`.
- **Step 1 fails with `The reserved System One endpoint must be typesafe/v1/systemone without a leading '/'.`**: the
  model's `supportedEndpoints` starts with a slash. Remove it.
- **Step 2 waits with `404`**: a new alias or model takes up to a minute to reach the runtime. The cell retries every
  15 seconds; rerun it if it timed out.
- **`401` with `authentication_error` and `Cannot authenticate with the server`**: TypeSafe rejected the key that the
  provider sends. Rerun step 0 and step 1 with a valid TypeSafe key: the deployment updates the provider.
- **`415` with `SystemOneEncodingUnsupported`**: the client compressed the request body. The TypeSafe SDK compresses
  bodies of 1 KB or more when zstd is available, so create the client with `compression=False`.
- **`429` in step 2, 3, or 4**: the key is over the rate limit from an earlier run. Wait for `Retry-After` seconds,
  then rerun the cell.
- **No `429` in step 5**: check that the alias lists the `requestRateLimit` policy in step 1, and that `small` sends
  the alias in `model`.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-31-jev-decisions).

## Next steps

- Add Anthropic and other Custom providers next to Jev in
  [Publish models from any provider behind stable aliases](../03-multi-provider-aliases/README.md).
- Give every caller a token allowance in [Throttle tokens per caller with token limits](../06-token-limits/README.md)
  and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- See decision requests and their tokens per caller in [Monitor usage, cost, and traces](../13-monitoring/README.md).
- Docs: [Use Jev decision models](https://ai.gateway.azure.com/docs/models/use-jev) and
  [Releases: November 9, 2026](https://ai.gateway.azure.com/docs/releases#release-2026-11-09).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/31-jev-decisions/](https://azure.github.io/aigateway/labs/31-jev-decisions/).